In [1]:
import cv2
import mediapipe as mp
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

In [2]:
mpPose = mp.solutions.pose
pose = mpPose.Pose()
mpDraw = mp.solutions.drawing_utils

In [9]:
video_path = 'P:/2023-0364-NeuroliveJazz/Camera and audio from study/Camera Recordings with timecode and brighter'
path_file: str = f"{video_path}/{'NeuroLive_Day1_ConcertA_Cam3.mp4'}"


In [4]:
# Select part of video sample that we want to analyze

cap = cv2.VideoCapture(path_file) 
fps = cap.get(cv2.CAP_PROP_FPS)

origin = "00:00:00"
start = "00:05:00" #00:05:10"
end = "00:10:00"

origintime = datetime.strptime(origin, "%H:%M:%S")
starttime = datetime.strptime(start, "%H:%M:%S")
endtime = datetime.strptime(end, "%H:%M:%S")

start_frame = fps*(starttime-origintime).total_seconds()
end_frame = fps*(endtime-origintime).total_seconds()

cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

True

In [5]:
# Selecting regions of interest and getting their coordinates

# Global variables to store the coordinates of the region
ix, iy, drawing = -1, -1, False
selections = []  # List of coordinates of multiple selected regions

# Mouse callback function
def select_multiple_regions(event, x, y, flags, param):
    global ix, iy, drawing, image, clone, selections

    if event == cv2.EVENT_LBUTTONDOWN:
        # Start drawing (first click)
        drawing = True
        ix, iy = x, y
    elif event == cv2.EVENT_MOUSEMOVE:
        # Update the rectangle while the mouse is moving
        if drawing:
            image = clone.copy()  # Reset the image to the original
            for (startX, startY, w, h) in selections:
                cv2.rectangle(image, (startX, startY), (startX + w, startY + h), (0, 255, 0), 2)  # Draw previously selected regions
            cv2.rectangle(image, (ix, iy), (x, y), (0, 255, 0), 2)  # Draw rectangle
    elif event == cv2.EVENT_LBUTTONUP:
        # Finish drawing (release the mouse button)
        drawing = False
        selections.append((ix, iy, x - ix, y - iy))  # Store the selected region (x, y, width, height)
        print(f"Selected Region: (x: {ix}, y: {iy}), Width: {x - ix}, Height: {y - iy}")

# Read the first frame
success, image = cap.read()
if not success:
    print("Error: Could not read the first frame.")
    cap.release()
    cv2.destroyAllWindows()
    exit()

clone = image.copy()  # Keep a copy of the original image

# Create a window and set the mouse callback function
cv2.namedWindow("Select Multiple Regions")
cv2.setMouseCallback("Select Multiple Regions", select_multiple_regions)

while True:
    # Display the image
    cv2.imshow("Select Multiple Regions", image)
    
    # Press 'q' to quit
    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break
    elif key == ord('r'):  # Press 'r' to reset selections
        selections = [] 
        image = clone.copy() 

cv2.destroyAllWindows()

Selected Region: (x: 1165, y: 368), Width: 349, Height: 424


In [6]:
# Function for processing with Mediapipe of individual ROI

def process_area_of_interest(cap, area_of_interest, n_frames):
    print(f"Vertices of area of interest: {area_of_interest}")
    results = {}

    # Define the landmark connections
    POSE_CONNECTIONS = mpPose.POSE_CONNECTIONS

    frame_count = 0

    while True:
        
        success, image = cap.read()

        if not success:
            return results
        
        if frame_count > n_frames:
            return results
        
        print(f"Processing frame number: {frame_count}")
        
        # drawing the bounding boxes
        x_shape, y_shape = image.shape[1], image.shape[0]
    
        x1, y1, x2, y2 = int(area_of_interest[0]), int(area_of_interest[1]), int(area_of_interest[0]+area_of_interest[2]), int(area_of_interest[1]+area_of_interest[3])
    
    
        # Ensure that the coordinates are within the image dimensions
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(x_shape, x2), min(y_shape, y2)

        # Crop the image based on the bounding box
        cropped_image = image[y1:y2, x1:x2]

        
        # MediaPipe Pose Prediction on the cropped region
        curr_results = pose.process(cropped_image)

        results[f'{frame_count}'] = curr_results
        frame_count +=1


In [7]:
max_frames = 200

all_results = []
for selection in selections:
    selection_result = process_area_of_interest(cap, selection, max_frames)
    all_results.append(selection_result)

Vertices of area of interest: (1165, 368, 349, 424)
Processing frame number: 0
Processing frame number: 1
Processing frame number: 2
Processing frame number: 3


c:\Users\andrea.gonzalez\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Processing frame number: 4
Processing frame number: 5
Processing frame number: 6
Processing frame number: 7
Processing frame number: 8
Processing frame number: 9
Processing frame number: 10
Processing frame number: 11
Processing frame number: 12
Processing frame number: 13
Processing frame number: 14
Processing frame number: 15
Processing frame number: 16
Processing frame number: 17
Processing frame number: 18
Processing frame number: 19
Processing frame number: 20
Processing frame number: 21
Processing frame number: 22
Processing frame number: 23
Processing frame number: 24
Processing frame number: 25
Processing frame number: 26
Processing frame number: 27
Processing frame number: 28
Processing frame number: 29
Processing frame number: 30
Processing frame number: 31
Processing frame number: 32
Processing frame number: 33
Processing frame number: 34
Processing frame number: 35
Processing frame number: 36
Processing frame number: 37
Processing frame number: 38
Processing frame number: 3

In [8]:
# Plotting landmarks and connections between them in the original video

# I added to option to only work with upper body for concert settings where some participants lower bodies might be occluded 
upper_body_indices = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]
lower_body_connections = [(23, 25),
(24, 26),
(25, 27),
(26, 28),
(27, 29),
(28, 30),
(28, 32),
(29, 31),
(30, 32),
(27, 31)]

frame_count = 0

while True:
    success, image = cap.read()
    
    if not success or frame_count > max_frames:
        break

    for i, tmp_selection_results in enumerate(all_results):
        results = tmp_selection_results
        selection = selections[i]
        x1, y1, x2, y2 = int(selection[0]), int(selection[1]), int(selection[0]+selection[2]), int(selection[1]+selection[3])
        
        #Draw landmarks on image
        if str(frame_count) in results and results[str(frame_count)].pose_landmarks is not None:
            for idx, landmark in enumerate(results[str(frame_count)].pose_landmarks.landmark):
                #if idx in upper_body_indices:
                # Scale the landmarks back to the original image dimensions
                landmark_x = int(landmark.x * (x2 - x1)) + x1
                landmark_y = int(landmark.y * (y2 - y1)) + y1
                cv2.circle(image, (landmark_x, landmark_y), 5, (0, 255, 0), 1)

        # Draw the connections between landmarks (using the predefined connections)
                for connection in mpPose.POSE_CONNECTIONS:
                    #if connection not in lower_body_connections:
                    start_idx, end_idx = connection
                    start_landmark = results[str(frame_count)].pose_landmarks.landmark[start_idx]
                    end_landmark = results[str(frame_count)].pose_landmarks.landmark[end_idx]

                    # Get the x, y coordinates for each landmark
                    start_x = int(start_landmark.x * (x2 - x1)) + x1
                    start_y = int(start_landmark.y * (y2 - y1)) + y1
                    end_x = int(end_landmark.x * (x2 - x1)) + x1
                    end_y = int(end_landmark.y * (y2 - y1)) + y1

                    # Draw a line between the two landmarks (with color and thickness)
                    cv2.line(image, (start_x, start_y), (end_x, end_y), (0, 255, 0), 2)
        
    cv2.imshow("Image", image)
    frame_count += 1

    k = cv2.waitKey(1) # frame rate can be changed by this parameter
    if k == 113: # if "q" is pressed on the keyboard, it quits the loop and the video window is closed
        break

cap.release()
cv2.destroyAllWindows()